In [ ]:
#Step 1: Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, countDistinct

In [ ]:
# Step 2: Create SparkSession
pyspark = (
    SparkSession.builder
    .appName("UserEngagementGoldLayer")
    .getOrCreate()
)

In [5]:
# Step 3: Read Silver Layer datasets
clean_users_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\users.csv"
)

clean_activity_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\user_activity.csv"
)

clean_content_df = pyspark.read.option("header", True).option("inferSchema", True).csv(
    r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\silver\netflix_titles.csv"
)

In [6]:
# Step 4: Select only required columns
activity_selected_df = clean_activity_df.select(
    "user_id",
    "show_id",
    "watch_minutes"
)

users_selected_df = clean_users_df.select(
    "user_id",
    "user_name",
    "country"
)

content_selected_df = clean_content_df.select(
    "show_id",
    "title",
    "show_type"
)
#In production, unnecessary columns are removed before joins to reduce memory usage and shuffle cost.

In [ ]:
# Step 5: Validate join keys
activity_selected_df.printSchema()
users_selected_df.printSchema()
content_selected_df.printSchema()
#Before joining, validate that user_id and show_id exist and have compatible data types.

root
 |-- user_id: integer (nullable = true)
 |-- show_id: string (nullable = true)
 |-- watch_minutes: integer (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)

root
 |-- show_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- show_type: string (nullable = true)



In [ ]:
# Step 6: Join user activity with users
activity_users_df = (
    activity_selected_df
    .join(
        users_selected_df,
        on="user_id",
        how="inner"
    )
)
# This join adds user profile information such as user_name and country to each activity record.

In [9]:
# Step 7: Join with Netflix content
activity_users_content_df = (
    activity_users_df
    .join(
        content_selected_df,
        on="show_id",
        how="inner"
    )
)
# This join adds content details such as title and show_type to each user activity record.

In [ ]:
# Step 8: Create user_summary_df
user_summary_df= (
    activity_users_content_df
    .groupBy("user_id", "user_name", "country")
    ).agg(spark_sum("watch_minutes").alias("total_watch_minutes"),
        countDistinct("show_id").alias("shows_watched")
        ). orderBy(col("total_watch_minutes").desc())
#The data is grouped at user level.
#For each user, Spark calculates:
#- total watch minutes
#- number of unique shows watched

In [11]:
# Step 9: Display final output
user_summary_df.show(20, truncate=False)

+-------+---------+---------+-------------------+-------------+
|user_id|user_name|country  |total_watch_minutes|shows_watched|
+-------+---------+---------+-------------------+-------------+
|138    |User_138 |Australia|4124               |22           |
|422    |User_422 |UK       |3468               |19           |
|461    |User_461 |USA      |3202               |17           |
|221    |User_221 |India    |3165               |18           |
|283    |User_283 |India    |3075               |16           |
|7      |User_7   |Germany  |2941               |19           |
|213    |User_213 |France   |2863               |15           |
|103    |User_103 |India    |2773               |16           |
|264    |User_264 |India    |2769               |14           |
|31     |User_31  |UK       |2756               |12           |
|70     |User_70  |Germany  |2696               |14           |
|292    |User_292 |Germany  |2686               |16           |
|233    |User_233 |France   |2680       

In [12]:
# Step 10: Validate final output
user_summary_df.printSchema()
user_summary_df.count()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- total_watch_minutes: long (nullable = true)
 |-- shows_watched: long (nullable = false)



500

In [ ]:
#validation
user_summary_df.filter(col("total_watch_minutes").isNull()).show()
user_summary_df.filter(col("shows_watched") <= 0).show()
#Final Gold Layer output should not have null metrics or invalid show counts.

+-------+---------+-------+-------------------+-------------+
|user_id|user_name|country|total_watch_minutes|shows_watched|
+-------+---------+-------+-------------------+-------------+
+-------+---------+-------+-------------------+-------------+

+-------+---------+-------+-------------------+-------------+
|user_id|user_name|country|total_watch_minutes|shows_watched|
+-------+---------+-------+-------------------+-------------+
+-------+---------+-------+-------------------+-------------+



In [ ]:
# import pandas to store the final output in a CSV file
import pandas as pd

In [ ]:
import os

gold_path = r"C:\Users\jovit\OneDrive\Documents\GitHub\data-engineering-90day-challenge-\01.Netflix Recommendation Engine\01.-data\03.curated\gold"

os.makedirs(gold_path, exist_ok=True)

user_summary_df.toPandas().to_csv(
    os.path.join(gold_path, "user_summary.csv"),
    index=False
)
#In production, Gold Layer would usually be saved as Parquet, Delta, or written into a data warehouse.
# In this local setup, CSV is used temporarily because Spark write requires Hadoop/winutils configuration on Windows.